# Window selection map

This notebook slows down the repo's new decision card: **which window should win once the task is specific enough that a single default answer becomes lazy?**

The point is not to produce one more global ranking. The point is to use the repo's existing metrics to build a bounded task map that can actually say different things for different jobs.


## 1. What goes into the map

The decision card reuses metrics that already exist in the repo:

- ENBW
- null-to-null main-lobe width
- peak sidelobe level
- half-bin scalloping loss
- quarter-hop synthesis-gain swing from the weighted overlap-add sidecar

That matters because the map is not a second world with made-up rules. It is a thin task layer over the same measurements the repo already exposes.


In [ ]:
from windowlab.recommend import TASK_PROFILES, build_task_metrics, build_task_rankings

rows = build_task_metrics()
rankings = build_task_rankings(rows)
[(row.name, round(row.enbw_bins, 3), round(row.main_lobe_width_bins, 2), round(row.scalloping_loss_db, 3)) for row in rows]


## 2. Why guardrails come before ranking

A weighted score without guardrails is still a trap.

If the task is amplitude honesty, a window with huge scalloping loss should not stay competitive just because it is narrow.
If the task is quarter-hop STFT framing, a window with a loud synthesis-normalization swing should not be called viable just because its sidelobes look nice.

So each task in the map does two things in order:

1. reject windows that obviously violate the lane's scope
2. rank the survivors with weighted versions of the repo's existing metrics


In [ ]:
[(task.key, task.max_values, task.min_values) for task in TASK_PROFILES]


## 3. Read the winners, not just the scores

The five top picks in the current map are:

- close equal-strength tones -> rectangular
- compact low-sidelobe compromise -> Kaiser `\beta = 8.6`
- weak spur beside a strong line -> Nuttall
- isolated-tone amplitude honesty -> flat-top
- quarter-hop STFT with calmer reconstruction -> Hamming

That spread is the real point. The repo now has enough evidence to stop pretending one answer covers all five jobs.


In [ ]:
for task in TASK_PROFILES:
    top = next(row for row in rankings[task.key] if row.eligible)
    print(task.key, '->', top.window, round(top.suitability, 3))


## 4. One column worth reading literally

The quarter-hop STFT lane is a good stress test because it is not a standard textbook ranking category.

It only exists because the repo already measured the weighted overlap bill separately from the raw overlap sum.
That lets the map say something sharper than 'Hann is common in STFTs.'

In this bounded framing setup, Hamming wins because it stays extremely calm on the synthesis-normalization metric while keeping better sidelobes and slightly lower ENBW than Hann.


In [ ]:
stft_rows = rankings['stft_qhop']
[(row.window, row.eligible, round(row.synthesis_gain_span_db, 4), round(row.peak_sidelobe_db, 1), round(row.score, 3)) for row in stft_rows[:5]]


## 5. What this notebook does not claim

This is not a universal best-window theorem.

It does not claim that:

- the guardrails are permanent for every application
- the same winners survive every frame length and FFT size
- the close-tone lane is safe once leakage becomes the main risk
- flat-top becomes a sane default just because it wins one column

It only claims that the repo's current metric packet is now strong enough to support a short task map without bluffing.


## 6. Best next move

If this lane gets one more pass, the strongest follow-up is probably not another ranking.
It is a bounded reconstruction note that asks when the same-window analysis/synthesis pair stays numerically calm after actual overlap-add normalization.

Useful companions:

- `notes/window-selection-map.md`
- `art/window-selection-map.png`
- `art/window-selection-map.csv`
- `notes/raw-overlap-is-not-the-synthesis-rule.md`
